In [1]:
# MODEL DETAILS ---------------------------------------------------------------------

# Training name
training_name = 'MaxConv2D_SQ_0e_2bit'

# Initial thresholds from transformer
initial_thresholds = [1,30,856]

# Gaussian noise parameters
NOISE_MU = 0.0
NOISE_SIGMA = 0.0 # e-

# Precision of input data
N_BITS = 2

In [2]:
# IMPORTS ---------------------------------------------------------------------

import warnings
warnings.filterwarnings("ignore")

import os
import random

import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, Callback
import csv

from DG.OptimizedDataGenerator_v2p5 import OptimizedDataGenerator
from loss import custom_loss
from SoftQuantizeLayer import SoftQuantizeLayer
from AnnealingScheduler import AnnealingScheduler

from models import *

2026-05-22 12:59:33.973068: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
pi = 3.14159265359
maxval=1e9
minval=1e-9

In [4]:
# TRAINING DATA ---------------------------------------------------------------------

dataset_base_dir = "/uscms/home/bweiss/nobackup/smart-pixels/"
tfrecords_base_dir = "/uscms/home/jennetd/nobackup/smart-pixels/tfrecords"

dataset_dir_train = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_centeredIncidence_parquets", 'train_contained/')
dataset_dir_val = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_centeredIncidence_parquets", 'test_contained/')

tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train",'3sr_16x16_'+str(int(NOISE_SIGMA))+'eNoise_train')
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val",'3sr_16x16_'+str(int(NOISE_SIGMA))+'eNoise_test')


In [5]:
print("Running training of model " + training_name)   
with open('log_'+training_name+'.txt','a') as f:
    f.write("Running training of model " + training_name + "\n")
    
seed = random.randint(0, 1000)
print("Seed: ", seed)
with open('log_'+training_name+'.txt','a') as f:
    f.write('Seed: ' + str(seed) + "\n")
    
# Loading pre-generated TFRecords
validation_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir= tfrecords_dir_val,
    shuffle=True,
    seed=seed,
    quantize=False,
)

training_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = tfrecords_dir_train,
    shuffle=True,
    seed=seed,
    quantize=False,
)
  

Running training of model MaxConv2D_SQ_0e_2bit
Seed:  611
Loading metadata from /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_val/3sr_16x16_0eNoise_test/metadata.json
Loading metadata from /uscms/home/jennetd/nobackup/smart-pixels/tfrecords/TFR_train/3sr_16x16_0eNoise_train/metadata.json


In [6]:
print("Initial thresholds: ", initial_thresholds)
with open('log_'+training_name+'.txt','a') as f:
    f.write("Initial thresholds: " + str(initial_thresholds) + "\n")

model = CreateSQModel(shape = (16,16,2), 
                      output = 14, 
                      n_filters=5,
                      pool_size=3,
                      initial_thresholds=initial_thresholds)

model.compile(
    optimizer=tf.keras.optimizers.Nadam(learning_rate=1e-3, clipnorm=1.0),
    loss=custom_loss,
)

fingerprint = '%08x' % random.randrange(16**8)

print("Fingerprint: ", fingerprint)
with open('log_'+training_name+'.txt','a') as f:
    f.write('Fingerprint: ' + str(fingerprint) + "\n")
    
base_dir = f'/uscms/home/jennetd/nobackup/smart-pixels/noise-paper/trained_models/model-{fingerprint}-{training_name}-checkpoints'
checkpoints_dir = os.path.join(base_dir, 'checkpoints')
checkpoint_filepath = os.path.join(checkpoints_dir, 'weights.{epoch:02d}-t{loss:.2f}-v{val_loss:.2f}.hdf5')
    
os.makedirs(base_dir, exist_ok=True)
os.makedirs(checkpoints_dir, exist_ok=True) 

Initial thresholds:  [1, 30, 856]
Fingerprint:  4027e1d1


In [7]:
early_stopping_patience = 500
es = EarlyStopping(patience=early_stopping_patience, restore_best_weights=True)

mcp = ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    save_freq='epoch'
)

csv_logger = CSVLogger(f'{base_dir}/training_log.csv', append=True)
scheduler_callback = AnnealingScheduler(
    schedule='cosine',  
    target_layer_name='soft_quantizer_output', 
    initial_k=1.0,
    final_k=67.0, 
    verbose=1      
)

quantizer_logger = SoftQuantizeLoggerCallback(
    log_filepath=f"{base_dir}/soft_quantizer_state_log.csv", # New, more descriptive filename
    layer_name="soft_quantizer_output"
)

In [ ]:
history = model.fit(
    x=training_generator,
    validation_data=validation_generator,
    callbacks=[es,mcp, csv_logger, scheduler_callback, quantizer_logger],
    epochs=5000,
    shuffle=True,
    verbose=0
)


Epoch 1: Annealing 'k' set to 1.0000
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
	Thresholds: 1.3133, 30.3133, 856.3135
	Tau: [ 15.156631 427.5001   826.0002  ]

Epoch 2: Annealing 'k' set to 1.0000
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
	Thresholds: 1.3421, 30.6326, 870.6569
	Tau: [ 15.316308 434.6574   840.0243  ]

Epoch 3: Annealing 'k' set to 1.0000
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
	Thresholds: 1.3456, 30.6312, 877.1370
	Tau: [ 15.315618 437.89566  846.50574 ]

Epoch 4: Annealing 'k' set to 1.0001
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
	Thresholds: 1.3583, 30.8216, 884.4821
	Tau: [ 15.410798 441.5619   853.66046 ]

Epoch 5: Annealing 'k' set to 1.0001
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
	Thresholds: 1.3969, 31.2038, 896.2807
	Tau: [ 15.601884 447.4419   865.0769  ]

Epoch 6: Annealing 'k' set to 1.0002
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
	Thresholds: 1.4126, 31.2972, 899.2006
	Tau: [ 15.648598 448.89398  867.9034  ]

Epoch 7: Annealing 'k' set to 1.0002
	Levels: 0.000